# 📊 EDA + Feature Engineering for EV Purchase Prediction

**Competition:** [Playground Series — Season 6, Episode 9](https://www.kaggle.com/competitions/playground-series-s6e9)

---

### 📋 What This Notebook Covers

| Section | Description |
|---------|-------------|
| 1 | **Data Overview** — Shape, types, missing values, duplicates |
| 2 | **Target Analysis** — Class balance, base rate |
| 3 | **Univariate Analysis** — Distribution of each feature by target |
| 4 | **Bivariate Analysis** — Feature interactions, correlation patterns |
| 5 | **Feature Engineering Ideas** — Derived features, encoding strategies |
| 6 | **Statistical Tests** — Feature importance via chi-squared & mutual info |

> 🎯 **Goal:** Deep understanding of the data to inform modeling decisions. This is a companion to the [CatBoost Ensemble submission notebook](https://www.kaggle.com/code/znidhal/).

---

In [ ]:
# ── Cell 1: Imports & Setup ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from scipy import stats
import pathlib, warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0e1117', 'axes.facecolor': '#0e1117',
    'axes.edgecolor': '#333', 'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0', 'xtick.color': '#aaa', 'ytick.color': '#aaa',
    'grid.color': '#222', 'font.size': 11, 'figure.figsize': (14, 5),
})
P = ['#00d2ff', '#ff6b6b', '#51cf66', '#ffd43b', '#cc5de8', '#ff922b', '#20c997', '#f783ac']

print('✅ Setup complete')

In [ ]:
# ── Cell 2: Load Data ────────────────────────────────────────────────────────────
ON_KAGGLE = pathlib.Path('/kaggle').exists()
DATA_DIR = pathlib.Path('/kaggle/input/playground-series-s6e9') if ON_KAGGLE else pathlib.Path('data')

def find_file(root, name):
    matches = sorted(root.glob(f'**/{name}'), key=lambda p: (len(p.parts), str(p)))
    if not matches:
        raise FileNotFoundError(f'{name} not found under {root}')
    return matches[0]

train = pd.read_csv(find_file(DATA_DIR, 'train.csv'))
test = pd.read_csv(find_file(DATA_DIR, 'test.csv'))

TARGET = 'Will_Buy_EV'
ID_COL = 'id'

print(f'🔢 Train: {train.shape[0]:,} rows × {train.shape[1]} columns')
print(f'🔢 Test:  {test.shape[0]:,} rows × {test.shape[1]} columns')
print(f'\n📋 Columns: {list(train.columns)}')
train.head()

## 📋 Section 1: Data Overview

In [ ]:
# ── Cell 3: Comprehensive Data Summary ───────────────────────────────────────────
def data_summary(df, name='DataFrame'):
    print(f'\n{"="*60}')
    print(f'📊 {name} Summary')
    print(f'{"="*60}')
    print(f'Shape: {df.shape}')
    print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    print(f'Duplicates: {df.duplicated().sum()}')
    
    summary = pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.notna().sum(),
        'null': df.isna().sum(),
        'null_pct': (df.isna().sum() / len(df) * 100).round(2),
        'nunique': df.nunique(),
        'sample_value': df.iloc[0],
    })
    display(summary)
    return summary

train_summary = data_summary(train, 'Train')
_ = data_summary(test, 'Test')

## 🎯 Section 2: Target Analysis

In [ ]:
# ── Cell 4: Target Distribution Deep-Dive ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = train[TARGET].value_counts()
pcts = train[TARGET].value_counts(normalize=True) * 100
bars = axes[0].bar(counts.index.astype(str), counts.values, color=[P[0], P[1]],
                   edgecolor='white', linewidth=0.5, width=0.5)
for bar, val, pct in zip(bars, counts.values, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=12, fontweight='bold', color='white')
axes[0].set_title(f'Target: {TARGET}', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=[f'Not Buy (0)\n{pcts.values[0]:.1f}%', f'Will Buy (1)\n{pcts.values[1]:.1f}%'],
           colors=[P[0], P[1]], startangle=90, explode=[0, 0.05],
           textprops={'color': 'white', 'fontsize': 12, 'fontweight': 'bold'},
           wedgeprops={'edgecolor': 'white', 'linewidth': 1})
axes[1].set_title('Class Balance', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

ratio = counts.min() / counts.max()
print(f'\nClass ratio: {ratio:.3f} (1.0 = perfectly balanced)')
print(f'Imbalance: {"⚠️ Moderate" if ratio < 0.3 else "✅ Acceptable" if ratio < 0.8 else "✅ Well-balanced"}')

## 📈 Section 3: Univariate Analysis

In [ ]:
# ── Cell 5: Numeric Feature Distributions by Target ──────────────────────────────
numeric_cols = train.select_dtypes(include=[np.number]).columns.drop([ID_COL, TARGET], errors='ignore')
cat_cols = train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features: {len(numeric_cols)}')
print(f'Categorical features: {len(cat_cols)}')

if len(numeric_cols) > 0:
    n_cols = 4
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    axes = axes.flatten()
    
    for idx, col in enumerate(numeric_cols):
        ax = axes[idx]
        for label, color in zip(sorted(train[TARGET].unique()), [P[0], P[1]]):
            subset = train[train[TARGET] == label][col].dropna()
            ax.hist(subset, bins=40, alpha=0.55, color=color, label=f'{TARGET}={label}', density=True)
        ax.set_title(col, fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.1)
    
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Numeric Feature Distributions by Target', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 6: Categorical Feature Analysis ─────────────────────────────────────────
if len(cat_cols) > 0:
    n_show = min(len(cat_cols), 8)
    n_cols_plot = min(4, n_show)
    n_rows_plot = (n_show + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(20, 5 * n_rows_plot))
    if n_rows_plot == 1 and n_cols_plot == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, col in enumerate(cat_cols[:n_show]):
        ax = axes[idx]
        # Target rate per category
        ct = train.groupby(col)[TARGET].agg(['mean', 'count']).sort_values('mean', ascending=True)
        ct = ct.tail(15)  # Show top 15 categories
        
        bars = ax.barh(ct.index.astype(str), ct['mean'], color=P[2], 
                       edgecolor='white', linewidth=0.3)
        ax.axvline(x=train[TARGET].mean(), color=P[1], linestyle='--', linewidth=1.5, 
                  label=f'Overall rate: {train[TARGET].mean():.3f}')
        ax.set_title(f'{col} (n_unique={train[col].nunique()})', fontsize=11, fontweight='bold')
        ax.set_xlabel(f'P({TARGET}=1)')
        ax.legend(fontsize=8)
    
    for idx in range(n_show, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Categorical Features — Target Rate per Category', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('No categorical features found.')

## 🔗 Section 4: Bivariate Analysis

In [ ]:
# ── Cell 7: Correlation Matrix ───────────────────────────────────────────────────
if len(numeric_cols) > 1:
    corr_cols = list(numeric_cols) + [TARGET]
    corr = train[corr_cols].corr()
    
    fig, ax = plt.subplots(figsize=(max(10, len(corr_cols)*0.8), max(8, len(corr_cols)*0.6)))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    cmap = sns.diverging_palette(220, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, center=0, vmin=-1, vmax=1,
                square=True, linewidths=0.5, annot=True, fmt='.2f', annot_kws={'size': 8}, ax=ax)
    ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.show()
    
    # High correlations
    high_corr = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            if abs(corr.iloc[i, j]) > 0.5:
                high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))
    
    if high_corr:
        print('\n⚠️ High correlations (|r| > 0.5):')
        for f1, f2, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
            print(f'   {f1:25s} ↔ {f2:25s} : r = {r:+.3f}')

In [ ]:
# ── Cell 8: Top Feature Pair Scatter Plots ───────────────────────────────────────
target_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
top_features = target_corr.head(4).index.tolist()

if len(top_features) >= 2:
    n_pairs = min(6, len(top_features) * (len(top_features)-1) // 2)
    fig, axes = plt.subplots(1, min(3, n_pairs), figsize=(20, 6))
    if not hasattr(axes, '__len__'):
        axes = [axes]
    
    pair_idx = 0
    for i in range(len(top_features)):
        for j in range(i+1, len(top_features)):
            if pair_idx >= len(axes):
                break
            ax = axes[pair_idx]
            f1, f2 = top_features[i], top_features[j]
            
            for label, color, marker in zip(sorted(train[TARGET].unique()), [P[0], P[1]], ['o', 's']):
                mask = train[TARGET] == label
                ax.scatter(train.loc[mask, f1], train.loc[mask, f2], 
                          c=color, alpha=0.2, s=10, marker=marker, label=f'{TARGET}={label}')
            
            ax.set_xlabel(f1, fontsize=10)
            ax.set_ylabel(f2, fontsize=10)
            ax.set_title(f'{f1} vs {f2}', fontsize=12, fontweight='bold')
            ax.legend(fontsize=9, markerscale=3)
            pair_idx += 1
    
    plt.suptitle('Top Feature Pairs — Colored by Target', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 🧪 Section 5: Feature Engineering Ideas

In [ ]:
# ── Cell 9: Feature Importance — Mutual Information ───────────────────────────────
# Encode categoricals for MI computation
X_mi = train[list(numeric_cols)].fillna(0)
y_mi = train[TARGET]

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)
mi_df = pd.DataFrame({'feature': numeric_cols, 'MI_score': mi_scores}).sort_values('MI_score', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, len(mi_df) * 0.4)))
colors_mi = [P[2] if s > mi_scores.mean() else P[0] for s in mi_df['MI_score']]
ax.barh(mi_df['feature'], mi_df['MI_score'], color=colors_mi, edgecolor='white', linewidth=0.3)
ax.axvline(x=mi_scores.mean(), color=P[1], linestyle='--', linewidth=1.5, 
          label=f'Mean MI: {mi_scores.mean():.4f}')
ax.set_title('Feature Importance — Mutual Information with Target', fontsize=14, fontweight='bold')
ax.set_xlabel('Mutual Information Score')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('\n🔑 Top 10 Features by Mutual Information:')
for _, row in mi_df.sort_values('MI_score', ascending=False).head(10).iterrows():
    print(f'   {row["feature"]:30s} | MI = {row["MI_score"]:.4f}')

In [ ]:
# ── Cell 10: Numeric Feature Statistics Summary ──────────────────────────────────
stats_by_target = []
for col in numeric_cols:
    for label in sorted(train[TARGET].unique()):
        subset = train[train[TARGET] == label][col]
        stats_by_target.append({
            'feature': col,
            'target': label,
            'mean': subset.mean(),
            'median': subset.median(),
            'std': subset.std(),
            'skew': subset.skew(),
            'kurtosis': subset.kurtosis(),
        })

stats_df = pd.DataFrame(stats_by_target)
print('📊 Feature Statistics by Target Class:')
display(stats_df.round(3))

In [ ]:
# ── Cell 11: Train vs Test Distribution Check ────────────────────────────────────
print('🔍 Train vs Test Distribution Comparison')
print('(Large KS statistic = distribution shift = potential problem)\n')

ks_results = []
for col in numeric_cols:
    if col in test.columns:
        stat, pval = stats.ks_2samp(train[col].dropna(), test[col].dropna())
        ks_results.append({'feature': col, 'KS_stat': stat, 'p_value': pval})

ks_df = pd.DataFrame(ks_results).sort_values('KS_stat', ascending=False)

fig, ax = plt.subplots(figsize=(10, max(5, len(ks_df) * 0.4)))
colors_ks = [P[1] if s > 0.1 else P[2] for s in ks_df['KS_stat']]
ax.barh(ks_df['feature'], ks_df['KS_stat'], color=colors_ks, edgecolor='white', linewidth=0.3)
ax.axvline(x=0.1, color='white', linestyle='--', alpha=0.5, label='Threshold (0.1)')
ax.set_title('Train vs Test Distribution Shift (KS Statistic)', fontsize=14, fontweight='bold')
ax.set_xlabel('KS Statistic')
ax.legend()
plt.tight_layout()
plt.show()

drift_features = ks_df[ks_df['KS_stat'] > 0.1]
if len(drift_features) > 0:
    print(f'\n⚠️ {len(drift_features)} features show distribution shift (KS > 0.1):')
    for _, row in drift_features.iterrows():
        print(f'   {row["feature"]:30s} | KS = {row["KS_stat"]:.4f}')
else:
    print('\n✅ No significant distribution shift detected between train and test!')

---

## 📌 Key Findings & Recommendations

| Finding | Implication |
|---------|-------------|
| Class balance | Determines if class weights or oversampling are needed |
| Feature correlations | Highly correlated features can be dropped or combined |
| Distribution shift | Features with train/test drift may need careful handling |
| Mutual information | Guides feature selection for model training |
| Categorical cardinality | High-cardinality features need special encoding |

### 🛠️ Feature Engineering Ideas
1. **Interaction features** — Multiply top correlated features
2. **Binning** — Convert continuous features to ordinal bins
3. **Target encoding** — Replace categories with target mean (with CV)
4. **Polynomial features** — Squared and cubed terms for non-linear relationships
5. **Frequency encoding** — Category count as a feature

---

**If this EDA was helpful, please upvote! 👍**